<a href="https://colab.research.google.com/github/rayaneghilene/PIDA-M2-Courses/blob/main/Lecture_4_Guided_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 4 — Introduction to data analysis with pandas

**Master 2 PIDA**  
**Dataset:** `political_posts.csv` (36 *synthetic* political posts; August-September 2026)

### Today's research question
**What topics appear in this small corpus, and how does the number of shares differ across topics and platforms?**

At the end you should be able to load and check a CSV, select and filter rows, count categories, compare groups, inspect the original text, and write one defensible finding with a limitation.

**How to work:** run cells from top to bottom with Shift+Enter. Each “Pause” asks you to predict or write an answer *before* running the next cell. You may edit and rerun a cell. We will use pandas commands as research tools; you do not need to memorize Python syntax.

## 0. Bring in your data

A **row** represents one post. A **column** describes a property of that post. This is a teaching dataset: the actors, posts and engagement numbers are synthetic. It cannot tell us what real voters or social media users did.

Download `political_posts.csv` and this notebook to your computer. In Colab, run the cell below and choose the CSV when prompted. When running locally, put the CSV next to the notebook.

In [ ]:
import pandas as pd

# Create a variable to store the path to your dataset
csv_path = "/content/political_posts.csv"

# Read the .csv file using pandas and assign it to the "df" variable
df = pd.read_csv(csv_path)

# Visualize the first 5 rows of your datasframe
df.head()

**Pause:** What is one unit of analysis? Is a row a person, an account, or a post? Which columns are words and which are counts?

## 1. Inspect before calculating

Start by checking size, column names, data types, blank cells, and duplicate rows. `df` is the whole table; `df["topic"]` is one column.

In [ ]:
# Print the number of the rows and columns of your dataframe using the .shape method
print("Rows and columns:", df.shape)

# Print the names of your columns
print("Column names:", df.columns.tolist())

# Check if you have duplicate rows in your dataframe using the .duplicated() method
print("Duplicate rows:", df.duplicated().sum())

# Print general information about your dataframe
df.info()

In [ ]:
# Chack if the number of missing values (we call them nan values) in each column
df.isna().sum()

**Pause:** Are there any empty cells? What type did pandas assign to `date`? Could you plot a genuine time trend without checking how the dates were sampled?

In [ ]:
df.sample(5, random_state=7)[["date", "actor", "country", "platform", "topic", "statement", "shares"]]

The first five rows are not a representative sample. Randomly inspecting several rows helps us catch surprising values and check whether our variables mean what we think they mean. Here the seed (`random_state=7`) makes the same five rows appear for everyone.

**Write one sentence:** What does `shares` measure, and what additional information would you need to compare *rates* of sharing?

## 2. Describe what is in the corpus

Count posts by topic, then by platform. These are *numbers of posts in this teaching file*, not estimates of how frequently these topics occur in politics generally.

In [ ]:
df["topic"].value_counts()

**Try it:** replace `"topic"` with `"platform"`. How many posts belong to each platform? Then repeat with `"country"`.

In [ ]:
df["platform"].value_counts()

**Pause:** Does the topic with the most posts necessarily receive the most shares per post? Why?

## 3. Select relevant evidence

Suppose we are interested in housing. A comparison should include the post texts, not just their numeric metrics. Read the command aloud: “keep rows where topic equals housing; show actor, country, statement and shares.”

In [ ]:
housing = df[df["topic"] == "housing"]
housing[["actor", "country", "statement", "shares"]]

**Try it:** show posts from Italy, displaying only `actor`, `topic`, and `statement`. Change the text inside the quotation marks; spelling and capitalization must match the column names and values.

Start with the scaffold below. Fill the three blanks and run it.

In [ ]:
# italy = df[df["country"] == "..."]
# italy[["...", "...", "..."]]

## 4. Compare groups

Our question is: **How many posts are in each topic, and how many shares does a typical post receive?** A mean is sensitive to unusually large values, so display a median as well. The `count` column matters because a mean based on four posts is fragile.

In [ ]:
topic_summary = (
    df.groupby("topic")["shares"]
      .agg(["count", "mean", "median"])
      .round(1)
      .sort_values("mean", ascending=False)
)
topic_summary

**Pause:** Which topic has the highest mean shares? How many posts support that figure? Does “highest in this file” mean “most popular among the public”?

In [ ]:
platform_summary = (
    df.groupby("platform")["shares"]
      .agg(["count", "mean", "median"])
      .round(1)
      .sort_values("mean", ascending=False)
)
platform_summary

**Try it:** compare `likes` across platforms by replacing `"shares"` in the previous cell. Do likes and shares give identical rankings?

## 5. Make one clear plot

A chart should name its measure and units. The bars below show the **mean number of shares per post**. They do not show total shares, audience size, or a share rate.

In [ ]:
import matplotlib.pyplot as plt

ax = platform_summary["mean"].plot.bar(
    color="#9CBBD5", edgecolor="#365776", figsize=(7, 4)
)
ax.set(title="Mean shares per post by platform (synthetic corpus)",
       xlabel="Platform", ylabel="Mean shares per post")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Pause:** Is this graph evidence that TikTok *causes* more sharing? The dataset has no audience size, impressions, or random assignment. What else might differ between the groups?

## 6. Test your first interpretation against the text

Averages can hide distinct messages and contexts. Look at the individual posts that contributed to one of the larger averages.

In [ ]:
df[df["platform"] == "TikTok"][["date", "actor", "topic", "statement", "shares"]]

Read two TikTok posts and two posts from another platform. What changes besides the platform? Do the messages, actors, or topics differ? Which single comparison would you want to investigate next?

In [ ]:
# Optional comparison: one topic across platforms.
# First run df["topic"].value_counts() and choose a topic.
chosen_topic = "work"
subset = df[df["topic"] == chosen_topic]
subset[["platform", "statement", "shares"]]

In [ ]:
subset.groupby("platform")["shares"].agg(["count", "mean"]).round(1)

**Pause:** Some cells contain only one or two posts. Would you put a strong conclusion about platforms in a paper based on those cells?

## 7. Write a research finding

Write 3-4 sentences using this structure:

1. **Question and scope:** “In this synthetic set of 36 posts, I compared …”
2. **Evidence:** report a number *and its denominator*, e.g. “TikTok posts averaged 88.6 shares across 9 posts; X posts averaged 50.1 across 8.”
3. **Interpretation:** “This is a descriptive difference in the sample.”
4. **Limitation and next data:** “To evaluate whether platform matters more broadly, I would need …”

**Do not write:** “TikTok makes political posts more shareable.” We cannot infer causation or population effects here.

For your exit ticket, submit your four sentences and the code cell you used as evidence.

---
### Reference / if you get stuck

- A `DataFrame` is the table; `df.head()` previews rows.
- `df["topic"]` selects a column; `df[["topic", "shares"]]` selects two.
- `df[df["country"] == "Italy"]` keeps rows satisfying a condition.
- `value_counts()` counts category labels; `groupby(...).agg(...)` summarizes numbers by category.
- `=` stores a result; `==` asks whether values are equal.
- Error: **FileNotFoundError** → upload the CSV or check its name. **KeyError** → check spelling and capitalization of a column or value. If `df` is undefined, rerun the loading cell.

**Further research issues:** origin and sampling of posts; selection bias; definitions of `topic`; whether annotation is consistent; observation time for engagement; platform audiences and exposure; ethics of collecting and quoting real posts. We will return to these when working with real corpora.